In [16]:
import os
import requests
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
import mlflow

DATA_PATH = "../../02_data/mnist"
NUM_CLIENTS = 3

mlflow.set_tracking_uri("http://localhost:5100")
MLFLOW_EXPERIMENT_NAME = "edge-fl-experiment"

In [3]:
!docker pull ghcr.io/bencefarkas27/aggregator:test-auth
!docker run --name aggregator_test -d --rm -p 5100:5000 -p 9100:8000  ghcr.io/bencefarkas27/aggregator:test-auth

test-auth: Pulling from bencefarkas27/aggregator
Digest: sha256:c5e22b76fdcc7fa313c3a07e40141c0296299ad2dabb38bbce3697098b784383
Status: Image is up to date for ghcr.io/bencefarkas27/aggregator:test-auth
ghcr.io/bencefarkas27/aggregator:test-auth
2365518f3b5c067c1298e02f2acc5ef4851f4f44eeea718b8ebc930cd0e5ce69


### Healt endpoint check

In [20]:
response = requests.get("http://localhost:9100/health")
print(f"Health check response of aggregator: {response.status_code} - {response.text}")

Health check response of aggregator: 200 - 


In [27]:
run_ids = []

def create_dummy_model(input_shape):
    model = keras.Sequential([
        keras.layers.Flatten(input_shape=input_shape),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def train_client(client_id):
    # Load client data
    x_train = np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_images.npy")
    y_train = np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_labels.npy")
    # Create and train the model
    model = create_dummy_model(input_shape=x_train.shape[1:])

    with mlflow.start_run(run_name=f"client_{client_id}_run") as run:
        mlflow.log_param("client_id", client_id)
        mlflow.log_param("num_samples", len(x_train))
        mlflow.log_param("input_shape", x_train.shape[1:])
        mlflow.log_param("num_classes", 10)

        model.fit(x_train, y_train, epochs=3)

        # Log the model to MLflow
        mlflow.tensorflow.log_model(model, f"client_{client_id}_model")

        run_ids.append(run.info.run_id)

    # Save the trained model weights
    model.save_weights(f"dummy_models/client_{client_id}.weights.h5")

In [28]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "admin"
with open("../../04_deploy/aggregator/cfg/mlflow_admin.json", "r") as f:
                admin_config = json.load(f)
os.environ["MLFLOW_TRACKING_PASSWORD"] = admin_config["admin_password"]
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
for client_id in range(NUM_CLIENTS):
    mlflow.sklearn.autolog()
    train_client(client_id)

/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/3
627/627 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8413 - loss: 6.5408
Epoch 2/3
627/627 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8893 - loss: 0.9741
Epoch 3/3
627/627 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9100 - loss: 0.4789


2026/07/22 11:18:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/22 11:18:23 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_0_run at: http://localhost:5100/#/experiments/1/runs/4f13868904004e6b8bb93896c78108fd
🧪 View experiment at: http://localhost:5100/#/experiments/1


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

🏃 View run client_1_run at: http://localhost:5100/#/experiments/1/runs/2dd62921f235486db1327913986eca6d
🧪 View experiment at: http://localhost:5100/#/experiments/1


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/3
623/623 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8231 - loss: 4.9832
Epoch 2/3
623/623 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8780 - loss: 0.6440
Epoch 3/3
623/623 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9095 - loss: 0.4118


2026/07/22 11:18:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/22 11:18:52 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_2_run at: http://localhost:5100/#/experiments/1/runs/1a029c93c45d4e30a7eb4cee712f2303
🧪 View experiment at: http://localhost:5100/#/experiments/1


### Calling do_aggregate

In [29]:
aggregation_config = {
    "local_model_hosts": [
        "http://localhost:5000"
        for _ in range(NUM_CLIENTS)
    ],
    "local_model_uris": [
        f"runs:/{run_ids[client_id]}/client_{client_id}_model"
        for client_id in range(NUM_CLIENTS)
    ],
    "local_data_size": [
        len(np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_images.npy"))
        for client_id in range(NUM_CLIENTS)
    ],
    "aggregation_algorithm": "fedavg",
    
}

response = requests.post("http://localhost:9100/aggregate_models", json=aggregation_config)

In [ ]:
print(response.json())

In [ ]:
# Pull the aggregated model from mlflow
try:
    # Search the experiment and retrieve the runs
    experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    # Get the latest run
    latest_run = runs.iloc[0]
    # Load the model from the latest run
    model = mlflow.sklearn.load_model(f"runs:/{latest_run.run_id}/model")

    print(f"Loaded model from run ID: {latest_run.run_id}")
except Exception as e:
    print(f"Failed to load model: {e}")

In [ ]:
# Testing the client models vs the aggregated model
# Load test data
x_test = np.load(f"{DATA_PATH}/test_images.npy")
y_test = np.load(f"{DATA_PATH}/test_labels.npy")

for client_id in range(NUM_CLIENTS):
    # Load the client model
    client_model = create_dummy_model(input_shape=x_test.shape[1:])
    client_model.load_weights(f"dummy_models/client_{client_id}.weights.h5")
    
    # Evaluate the client model
    client_loss, client_accuracy = client_model.evaluate(x_test, y_test, verbose=0)
    print(f"Client {client_id} Model - Loss: {client_loss}, Accuracy: {client_accuracy}")

In [8]:
!docker stop aggregator_test

aggregator_test
